In [ ]:
import os
import pandas as pd
import numpy as np

from pathlib import Path
project_path = Path.cwd()
export_path = project_path / "export"

sourcefile = r"C:\Users\jaychenghg\assessments\hdb\export\hdb_resale_master_with_source.csv"
dataframe = pd.read_csv(sourcefile)
# dataframe

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\jaychenghg\\assessments\\hdb\\export\\hdb_resale_master_with_source.csv'

# Data Quality Requirements
Part 1 has been completed in a different notebook that fetches the data, cleans them and merge them


## 1. Data Profiling


In [ ]:
# print(f"Rows: {dataframe.shape[0]:,}")
# print(f"Columns: {dataframe.shape[1]:,}")
# # -------------------------------------------------------------------------
# # Dataset structure
# # -------------------------------------------------------------------------
# dataframe.info()
# dataframe["town"].unique()
# dataframe["flat_type"].unique()

# sorted(dataframe["storey_range"].unique())
# -------------------------------------------------------------------------
# Profile Storey Range by transaction period
# -------------------------------------------------------------------------
# storey_profile = (
#     dataframe_filtered
#     .groupby("storey_range")
#     .agg(
#         min_month=("month", "min"),
#         max_month=("month", "max"),
#         record_count=("storey_range", "size")
#     )
#     .sort_values("storey_range")
# )

# storey_profile
# sorted(dataframe["lease_commence_date"].unique())


In [ ]:
# -------------------------------------------------------------------------
# Standardise Data for multi gen
# -------------------------------------------------------------------------
dataframe["flat_type"] = dataframe["flat_type"].replace({
    "MULTI-GENERATION": "MULTI GENERATION"
})

In [ ]:
dataframe["month_date"] = pd.to_datetime(
    dataframe["month"],
    format="%Y-%m",
    errors='coerce'
)
# display(dataframe)
dataframe_filtered = dataframe[(dataframe["month_date"] >= "2012-01-01") & (dataframe["month_date"] <= "2016-12-31")].copy()
display(dataframe_filtered.groupby("month").size().reset_index(name="record_count").head())

In [ ]:
print(f"Rows: {dataframe_filtered.shape[0]:,}")
print(f"Columns: {dataframe_filtered.shape[1]:,}")
# -------------------------------------------------------------------------
# Dataset structure
# -------------------------------------------------------------------------
dataframe_filtered.info()

In [ ]:
# -------------------------------------------------------------------------
# Column profiling
# -------------------------------------------------------------------------

profile = pd.DataFrame({
    "data_type": dataframe_filtered.dtypes.astype(str),
    "non_null_count": dataframe_filtered.notna().sum(),
    "null_count": dataframe_filtered.isna().sum(),
    "null_pct": (dataframe_filtered.isna().mean() * 100).round(2),
    "unique_count": dataframe_filtered.nunique(dropna=True),
    "unique_pcnt": (
        dataframe_filtered.nunique(dropna=True)/len(dataframe_filtered) * 100
    )
})

display(profile)

In [ ]:
# -------------------------------------------------------------------------
# Numeric profiling
# -------------------------------------------------------------------------
numeric_columns = [
    'floor_area_sqm',
    'lease_commence_date',
    'resale_price'
]

for col in numeric_columns:
    if col in dataframe_filtered.columns:
        dataframe_filtered[col] = pd.to_numeric(dataframe_filtered[col], errors='coerce')

display(dataframe_filtered[numeric_columns].describe().T)

## 2. Remaining Lease Recalculation

### Assumptions

- All HDB leases are assumed to have a 99-year lease period.
- As only the lease commencement year is available consistently, the lease commencement date is assumed to be 1 January of the corresponding year.
- Remaining lease is calculated as of the current system date.
- Remaining lease is rounded down to complete years and months.

In [ ]:
# -------------------------------------------------------------------------
# Recompute Remaining Lease
# sorted(dataframe_filtered["lease_end_date"].unique())

# As only the lease commencement year is available consistently in the master dataset, 
# 1 January of the lease commencement year is assumed as the lease start date. 
# Remaining lease is calculated against the current system date based on a 99-year lease 
# and rounded down to complete years and months.
# -------------------------------------------------------------------------

dataframe_filtered["lease_commence_date_act"] = pd.to_numeric(
    dataframe_filtered["lease_commence_date"],
    errors='coerce'
)

dataframe_filtered["lease_commence_date_act"] = pd.to_datetime(
    dataframe_filtered["lease_commence_date_act"].astype("Int64").astype(str)+"-01-01",
    errors='coerce'
)

dataframe_filtered["lease_end_date"] = dataframe_filtered["lease_commence_date_act"] + pd.DateOffset(years=99)

today = pd.Timestamp.today().normalize()
dataframe_filtered["remaining_months"] = dataframe_filtered["lease_end_date"] - today



In [ ]:
# -------------------------------------------------------------------------
# Recompute Remaining Lease
# sorted(dataframe_filtered["lease_end_date"].unique())

# As only the lease commencement year is available consistently in the master dataset, 
# 1 January of the lease commencement year is assumed as the lease start date. 
# Remaining lease is calculated against the current system date based on a 99-year lease 
# and rounded down to complete years and months.
# -------------------------------------------------------------------------

dataframe_filtered["lease_commence_date_act"] = pd.to_numeric(
    dataframe_filtered["lease_commence_date"],
    errors='coerce'
)

dataframe_filtered["lease_commence_date_act"] = pd.to_datetime(
    dataframe_filtered["lease_commence_date_act"].astype("Int64").astype(str)+"-01-01",
    errors='coerce'
)

dataframe_filtered["lease_end_date"] = dataframe_filtered["lease_commence_date_act"] + pd.DateOffset(years=99)

today = pd.Timestamp.today().normalize()
dataframe_filtered["remaining_months"] = dataframe_filtered["lease_end_date"] - today



In [ ]:
# -------------------------------------------------------------------------
# Recompute Remaining Lease
# sorted(dataframe_filtered["lease_end_date"].unique())

# Since only the lease commencement year is available consistently in the master dataset, 
# 1 January of the lease commencement year is assumed as the lease start date. 
# Remaining lease is calculated against the current system date based on a 99-year lease 
# and rounded down to complete years and months.
# -------------------------------------------------------------------------

dataframe_filtered["lease_commence_date_act"] = pd.to_numeric(
    dataframe_filtered["lease_commence_date"],
    errors='coerce'
)

dataframe_filtered["lease_commence_date_act"] = pd.to_datetime(
    dataframe_filtered["lease_commence_date_act"].astype("Int64").astype(str)+"-01-01",
    errors='coerce'
)

dataframe_filtered["lease_end_date"] = dataframe_filtered["lease_commence_date_act"] + pd.DateOffset(years=99)

today = pd.Timestamp.today().normalize()
dataframe_filtered["remaining_months"] = (
    (dataframe_filtered["lease_end_date"].dt.year - today.year) * 12 +
    (dataframe_filtered["lease_end_date"].dt.month - today.month)
)

dataframe_filtered["remaining_months"] = dataframe_filtered["remaining_months"] - (
    dataframe_filtered["lease_end_date"].dt.day < today.day
).astype(int)

dataframe_filtered["remaining_months"] = dataframe_filtered["remaining_months"].clip(lower=0)
dataframe_filtered["remaining_lease"] = (
    (dataframe_filtered["remaining_months"] // 12).astype("Int64").astype(str) + ' years ' +
    (dataframe_filtered["remaining_months"] % 12).astype("Int64").astype(str) + ' months'
)

In [ ]:
dataframe_filtered[["lease_commence_date", "lease_commence_date_act", "lease_end_date", "remaining_months", "remaining_lease"]].head()

## 3. Check for and remove duplicates, store duplicates in a separate dataframe


In [ ]:
# -------------------------------------------------------------------------
# Identify composite key duplicates
# -------------------------------------------------------------------------

# Technical and derived fields are excluded from the business composite key
exclude_columns = [
    "_id",
    "resale_price",
    "source_dataset_id",
    "source_dataset_name",
    "month_date",
    "lease_commence_date_act",
    "lease_end_date",
    "remaining_months"
]

composite_key = [
    col
    for col in dataframe_filtered.columns
    if col not in exclude_columns
]

print(composite_key)

# -------------------------------------------------------------------------
# Duplicate composite keys
# -------------------------------------------------------------------------
duplicate_key_mask = dataframe_filtered.duplicated(
    subset=composite_key,
    keep=False
)

print(f"Records with duplicated composite key: {duplicate_key_mask.sum():,}")

duplicate_key_records = dataframe_filtered[duplicate_key_mask].sort_values(composite_key + ["resale_price"])
# display(duplicate_key_records)

# -------------------------------------------------------------------------
# Convert resale price columns to numeric
# -------------------------------------------------------------------------
dataframe_filtered["resale_price"] = pd.to_numeric(
    dataframe_filtered["resale_price"],
    errors='coerce'
)

In [ ]:
# -------------------------------------------------------------------------
# Generate duplicated records and separate them from main data set
# -------------------------------------------------------------------------
dataframe_filtered_dedupe = dataframe_filtered.sort_values(
    by=composite_key + ["resale_price"],
    ascending=[True] * len(composite_key) + [False]
).copy()

dataframe_filtered_failed_mask = dataframe_filtered_dedupe.duplicated(
    subset=composite_key,
    keep="first"
)

# Failed records
dataframe_filtered_failed = dataframe_filtered_dedupe[dataframe_filtered_failed_mask].copy()
dataframe_filtered_failed["Reason for failing"] = ("Duplicates based on the composite key with a lower resale price")

# Records with higher resale price, to keep
dataframe_filtered_dedupe = dataframe_filtered_dedupe[~dataframe_filtered_failed_mask].copy()

In [ ]:
# -------------------------------------------------------------------------
# Verify duplicate pass and fail checks
# -------------------------------------------------------------------------
print(f"{'Number of records that before duplicates check':<80}: {len(dataframe_filtered_dedupe)}")
print(f"{'Number of records that failed the duplicates check (lower resale price)':<80}: {len(dataframe_filtered_failed)}")
print(f"{'Number of records that passed the duplicates check (higher resale price)':<80}: {len(dataframe_filtered_dedupe)}")
print(f"{'Duplicate composite keys remaining':<80}: {dataframe_filtered_dedupe.duplicated(subset=composite_key,keep=False).sum()}")

## 4. Resale Price Anomaly Detection

Resale prices are analysed within comparable transaction groups rather
than against the entire historical dataset, as resale prices vary over
time, location and flat characteristics.

Potential anomalies are flagged for further investigation rather than
automatically treated as invalid records.

Transactions are compared against transactions from the same month, town, flat type and 5-square-metre floor-area band. A resale price falling more than three standard deviations from the group mean is flagged as a potential anomaly.

Resale price distributions were assessed for skewness within comparable transaction groups. The median skewness was 0.35 and mean skewness was 0.43, indicating mild positive skew overall. As the degree and direction of skew varied across groups, a consistent ±3 standard deviation threshold was retained as a conservative heuristic for identifying potentially anomalous resale prices.


In [ ]:

dataframe_filtered_dedupe["area_band"] = (
    (dataframe_filtered_dedupe["floor_area_sqm"] // 5 * 5).astype("int").astype(str) + " -> " +
    ((dataframe_filtered_dedupe["floor_area_sqm"] // 5 * 5) + 4).astype("int").astype(str)
)
# display(
#     dataframe_filtered_dedupe
#     .groupby("area_band")
#     .size()
#     .reset_index()
# )

In [ ]:
stats_grp = ['month', 'town', 'flat_type', 'area_band']
price_stats = dataframe_filtered_dedupe.groupby(stats_grp)["resale_price"].agg(    
    mean="mean",
    median="median",
    std_dev="std",
    skewness="skew",
    record_count="size"
).reset_index()

# -------------------------------------------------------------------------
# Calculate lower and upper resale price thresholds
# -------------------------------------------------------------------------
price_stats["lower"] = (price_stats["mean"] - (price_stats["std_dev"] * 3))
price_stats["upper"] = (price_stats["mean"] + (price_stats["std_dev"] * 3))
# display(price_stats)
# display(price_stats[price_stats["record_count"]>=10]["skewness"].describe())

In [ ]:
price_stats.isna().sum()

In [ ]:
dataframe_filtered_stats = dataframe_filtered_dedupe.merge(price_stats, on=stats_grp, how="left")
# -------------------------------------------------------------------------
# Flag resale price outliers
# -------------------------------------------------------------------------
resale_price_outlier_mask = (
    (dataframe_filtered_stats["resale_price"] > dataframe_filtered_stats["upper"]) |
    (dataframe_filtered_stats["resale_price"] < dataframe_filtered_stats["lower"])    
)

resale_price_outliers = dataframe_filtered_stats[resale_price_outlier_mask].copy()
drop_cols = [
    'area_band', 
    'mean', 
    'median',
    'std_dev', 
    'skewness', 
    'record_count', 
    'lower', 
    'upper'    
]

resale_price_outliers = resale_price_outliers.drop(
    columns=drop_cols,
    errors="ignore"
)
resale_price_outliers["Reason for failing"] = ("Potential anomalies in resale price that are more than or lesser than 3 times standard deviations")

# -------------------------------------------------------------------------
# Resale price outlier summary
# -------------------------------------------------------------------------
outlier_count = resale_price_outlier_mask.sum()
total_count = len(dataframe_filtered_dedupe)
outlier_pcnt = (outlier_count/total_count) * 100

print(f"{'Total no. of units':<40} : {total_count:,} ")
print(f"{'No. of units flagged as outliers':<40} : {outlier_count:,} ")
print(f"{'Percentage of units flagged as outliers':<40} : {outlier_pcnt:.3f}%")


## 5. Final datasets and their count

In [ ]:
# -------------------------------------------------------------------------
# Generate final dataset (pass)
# -------------------------------------------------------------------------
dataframe_final = dataframe_filtered_stats[
    ~resale_price_outlier_mask
].copy()

# -------------------------------------------------------------------------
# Remove statistical helper columns
# -------------------------------------------------------------------------
stats_columns = [
    "mean_price",
    "std_price",
    "record_count",
    "lower",
    "upper"
]

dataframe_final = dataframe_final.drop(
    columns=stats_columns,
    errors="ignore"
)

print(f"Total no. of qualified dataset : {len(dataframe_final):,} ")


In [ ]:
# -------------------------------------------------------------------------
# Generate failed final dataset
# -------------------------------------------------------------------------
dataframe_failed = pd.concat(
    [
        resale_price_outliers, 
        dataframe_filtered_failed
    ], 
    ignore_index=True, 
    sort=False
)
# display(dataframe_failed.groupby("Reason for failing").size())

# -------------------------------------------------------------------------
# Final dataset validation
# -------------------------------------------------------------------------

print(f"{'Final successful records':<40}: {len(dataframe_final):,}")
print(f"{'Failed records':<40}: {len(dataframe_failed):,}")
print(f"{'Duplicate failures':<40}: {len(dataframe_filtered_failed):,}")
print(f"{'Price anomaly failures':<40}: {len(resale_price_outliers):,}")

In [ ]:
# pd.set_option("display.max_rows", None)
# pd.reset_option("display.max_rows")

# inspect outliers
# display(
#     resale_price_outliers[
#         [
#             "month",
#             "town",
#             "flat_type",
#             "area_band",
#             "resale_price",
#             "mean",
#             "std_dev",
#             "lower",
#             "upper",
#             "record_count"
#         ]
#     ].sort_values("resale_price", ascending=False)
# )

## 6. Other additional data validations

In [ ]:
# -------------------------------------------------------------------------
# Validate numeric values for floor area and resale price
# -------------------------------------------------------------------------
print(f"{'Invalid floor area':<40}: {(dataframe_final["floor_area_sqm"] <= 0).sum()}")
print(f"{'Invalid resale price':<40}: {(dataframe_final["resale_price"] <= 0).sum()}")
print("="*44)
# -------------------------------------------------------------------------
# Validate lease commencement year
# -------------------------------------------------------------------------
current_year = pd.Timestamp.today().year
invalid_lease_mask = (dataframe_final["lease_commence_date"] > current_year)
print(f"{'Invalid lease commencement dates':40}: {invalid_lease_mask.sum():,}")
print("="*44)
# -------------------------------------------------------------------------
# Validate transaction date
# -------------------------------------------------------------------------
today = pd.Timestamp.today().normalize()
future_transaction_mask = (dataframe_final["month_date"] > today)
print(f"{'Future transaction dates':<40}: {future_transaction_mask.sum():,}")
print("="*44)
# -------------------------------------------------------------------------
# Validate remaining lease
# -------------------------------------------------------------------------
invalid_remaining_lease_mask = (
    (dataframe_final["remaining_months"] < 0) |
    (dataframe_final["remaining_months"] > 99 * 12)
)
print(f"{'Invalid remaining lease':<40}: {invalid_remaining_lease_mask.sum():,}")
print("="*44)
# -------------------------------------------------------------------------
# Validate leading and trailing whitespaces
# -------------------------------------------------------------------------
categorical_columns = [
    "town",
    "flat_type",
    "flat_model",
    "storey_range"
]
for column in categorical_columns:
    whitespace_count = (dataframe_final[column] != dataframe_final[column].str.strip()).sum()
    print(f"{column:<40}: {whitespace_count:,}")
print("="*44)

In [ ]:
# -------------------------------------------------------------------------
# Combine validation failure masks
# -------------------------------------------------------------------------
validation_failed_mask = (
    (dataframe_final["floor_area_sqm"] <= 0) |
    (dataframe_final["resale_price"] <= 0) |
    invalid_lease_mask | 
    future_transaction_mask |
    invalid_remaining_lease_mask
)

for column in categorical_columns:
    validation_failed_mask = validation_failed_mask | (
        dataframe_final[column] != dataframe_final[column].str.strip()
    )

# -------------------------------------------------------------------------
# Store failed validation records
# -------------------------------------------------------------------------

dataframe_validation_failed = dataframe_final[
    validation_failed_mask
].copy()

def get_validation_failed_reason(row):
    reasons = []
    if row["floor_area_sqm"] <= 0:
        reasons.append("Invalid floor area")
    
    if row["resale_price"] <= 0:
        reasons.append("Invalid resale price")
    
    if row["lease_commence_date"] > current_year:
        reasons.append("Invalid lease commencement date")
    
    if row["month_date"] > today:
        reasons.append("Future transaction date")
    
    if (
        (row["remaining_months"] < 0) |
        (row["remaining_months"] > 99 * 12)        
    ):
        reasons.append("Invalid remaining lease")
    
    for column in categorical_columns:
        if row[column] != row[column].str.strip():
            reasons.append(f"Leading/Trailing whitespace(s) found in {column}")
        
    return "; ".join(reasons)



if len(dataframe_validation_failed) != 0:    
    dataframe_validation_failed["Reasons for failing"] = (
        dataframe_validation_failed.apply(
            get_validation_failed_reason,
            axis=1
        )
    )

    dataframe_failed = pd.concat(
        [
            dataframe_failed,
            dataframe_validation_failed        
        ],
        ignore_index=True,
        sort=False    
    )

# -------------------------------------------------------------------------
# Retain those records that has passed all the validations
# -------------------------------------------------------------------------
dataframe_final = dataframe_final[~validation_failed_mask].copy()

print(f"{'Validation failures added':<40}: {len(dataframe_validation_failed):,}")
print(f"{'New total failed records':<40}: {len(dataframe_failed):,}")
print(f"{'Remaining valid records':<40}: {len(dataframe_final):,}")

# Data Transformation Requirements

In [ ]:
print(f"{'Duplicate records':<40}: {dataframe_final.duplicated().sum():,}")

# -------------------------------------------------------------------------
# Average resale price by month, town and flat_type
# -------------------------------------------------------------------------
# print(dataframe_final.columns)
avg_group = [
    'month',
    'town', 
    'flat_type'
]

avg_resale_price = (
    dataframe_final
    .groupby(avg_group)["resale_price"]
    .mean()
    .reset_index(name="avg_resale_price")
)

# display(avg_resale_price)
dataframe_transformed = dataframe_final.merge(avg_resale_price, on=avg_group, how="left")

# -------------------------------------------------------------------------
# Derive transformed block
# -------------------------------------------------------------------------
dataframe_transformed["block_transformed"] = (
    dataframe_transformed["block"]
    .astype(str) # convert to string
    .str.replace(r"\D", "", regex=True) # Remove non digits
    .str[:3] # Only take the 1st 3 digits
    .str.zfill(3) # Prepend with 0s if its lesser than 3 digits
)

# -------------------------------------------------------------------------
# Derive resale price id
# -------------------------------------------------------------------------
dataframe_transformed["avg_resale_transformed"] = (
    dataframe_transformed["avg_resale_price"]
    .astype(int) # convert to integer
    .astype(str) # convert to string
    .str[:2] # Only take the 1st 2 digits
)

# -------------------------------------------------------------------------
# Derive the month id
# -------------------------------------------------------------------------
dataframe_transformed["month_transformed"] = (
    pd.to_datetime(dataframe_transformed["month"]).dt.strftime("%m")
)

# -------------------------------------------------------------------------
# Derive the town initials
# -------------------------------------------------------------------------
dataframe_transformed["town_transformed"] = (
    dataframe_transformed["town"]
    .astype(str)
    .str[0]
)

# -------------------------------------------------------------------------
# Create the resale identified based on the transformed columns
# -------------------------------------------------------------------------
dataframe_transformed["Resale Identifier"] = (
    "S" + 
    dataframe_transformed["block_transformed"] + 
    dataframe_transformed["avg_resale_transformed"] + 
    dataframe_transformed["month_transformed"] + 
    dataframe_transformed["town_transformed"]
)

# dataframe_transformed.head()

In [ ]:
# display(dataframe_transformed.groupby(["town", "town_transformed"]).size())
# display(dataframe_transformed.groupby(["month", "month_transformed"]).size())
# -------------------------------------------------------------------------
# Validate resale id uniqueness
# -------------------------------------------------------------------------
resale_identifier_duplicate_mask = (
    dataframe_transformed["Resale Identifier"]
    .duplicated(keep=False)
)

print(f"{'Duplicate Resale Identifiers':<40}: {resale_identifier_duplicate_mask.sum():,}")

### Caveat
The prescribed Resale Identifier is not inherently unique because several components are lossy transformations. In particular, only the town initial and transaction month are retained, while the transaction year is omitted. Consequently, distinct valid transactions may derive the same Resale Identifier. SHA-256 hashing is deterministic and therefore preserves these existing duplicate identifiers rather than resolving them. Hash uniqueness is therefore assessed relative to the uniqueness of the source Resale Identifier.

In [ ]:
import hashlib
dataframe_hashed = dataframe_transformed.copy()
dataframe_hashed["Hashed Resale Identifier"] = (
    dataframe_hashed["Resale Identifier"]
    .apply(lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest())
)

# display(dataframe_hashed.head())

### Export datasets in accordance to the following requirements
	- Raw: Contains the raw data files as-it-is. (dataframe_filtered)
	- Cleaned: Dataset that passes the data quality requirements (dataframe_final)
	- Transformed: Dataset that went through the transformation requirements (dataframe_transformed)
	- Failed: Dataset that contains records that were removed (E.g. Duplicated Records, Data that does not comply with any validation rules) 
    (dataframe_failed)
	- Hashed: Cleaned Data + Hashed Identifier Column (dataframe_hashed)

In [ ]:
print(f"{'Number of records before filtering validations':60} : {len(dataframe_filtered)}")
print(f"{'Number of failed records':60} : {len(dataframe_failed)}")
print(f"{'Number of final records before transformations':60} : {len(dataframe_final)}")
print(f"{'Number of final records after transformations':60} : {len(dataframe_transformed)}")
print(f"{'Number of hashed records':60} : {len(dataframe_hashed)}")


In [ ]:

dataframe_filtered.to_csv(export_path / "raw_data_before_validations.csv")
dataframe_final.to_csv(export_path / "final_data_before_transformations.csv")
dataframe_transformed.to_csv(export_path / "final_data_after_transformations.csv")
dataframe_failed.to_csv(export_path / "failed_data.csv")
dataframe_hashed.to_csv(export_path / "final_data_after_hash.csv")